# 1.消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()


# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"


# 创建Agent
agent = create_agent(model="deepseek-chat", tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("北京今天天气如何？"),
    ]
})
print(response)

## invoeke() 返回美化打印

In [ ]:
from rich import print as rprint
import json

# 方法一：rich 美化打印（结构高亮、缩进清晰）
rprint(response)

In [ ]:
# 方法二：json.dumps 序列化
print(json.dumps(response, indent=2, ensure_ascii=False, default=str))

In [ ]:
# 方法三：逐条消息 pretty_print()（人类最易读）
for message in response['messages']:
    message.pretty_print()

In [ ]:
# 方法四：查看 AIMessage 的元数据（token 用量、模型名等）

for message in response['messages']:
    if hasattr(message, 'response_metadata') and message.response_metadata:
        rprint(message.response_metadata)

# 2.多模态消息

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- qwen-vl-max（通义千问视觉模型）
- gpt-5-nano
- ...

我们以 qwen-vl-max 为例，演示向模型发送图片消息

## 2.1 Images

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import os
from dotenv import load_dotenv

load_dotenv()
# 初始化模型
model = init_chat_model(
    model="qwen-vl-plus",  # 通义千问多模态模型，支持图片、文本
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)
# 创建Agent
agent = create_agent(model=model)

In [ ]:
from tools import img2base64

base64_string = img2base64("resources/02.png")

print(base64_string)

In [ ]:
# URL
msg1 = HumanMessage([
    {
        "type": "text",
        "text": "描述一下这张图片的内容"
    },
    {
        "type": "image",
        "url": "https://pic.nximg.cn/file/20220904/7090656_112421454127_2.jpg"
    }
])

# base64 data
msg2 = HumanMessage([
    {
        "type": "text",
        "text": "描述一下这张图片的内容"
    }, {
        "type": "image",
        "base64": f"{base64_string}",
        "mime_type": "image/png"
    }
])

In [ ]:
resp = agent.invoke({"messages": [msg2]})

In [ ]:
from rich import print as rprint

rprint(resp['messages'][-1].content)

## 2.2 PDF （国内模型不支持）

In [ ]:
from tools import img2base64

base64_string = img2base64("resources/huwenxin.pdf")

print(base64_string)

In [ ]:
# URL
msg = HumanMessage([
    {
        "type": "text",
        "text": "描述一下这张图片的内容"
    },
    {
        "type": "file",
        "base64": f"{base64_string}",
        "mime_type": "application/pdf",
        "filename": "huwenxin.pdf"
    }
])

In [ ]:
resp = agent.invoke({"messages": [msg]})

In [ ]:
from rich import print as rprint

rprint(resp['messages'][-1].content)

## 2.3 Audio 音频识别（LangChain 风格）
- 使用 Qwen-Omni 全模态模型，支持**音频输入 + 文本输出**
- 通过 `init_chat_model` + `HumanMessage` 多模态消息调用，与前面 Images 章节风格统一
- model="qwen3.5-omni-plus"
- `model_kwargs={"modalities": ["text"]}` — 仅输出文本识别结果
- 音频以 base64 编码通过 `audio_url` 传入消息
- `model.stream()` 流式输出识别文本

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# ============================================================
# Qwen-Omni 音频识别（DashScope 原生 API）
# OpenAI 兼容端点不支持音频作为输入 content 类型，
# 只有原生 MultiModalConversation 接口支持
# ============================================================
import dashscope
from http import HTTPStatus

dashscope.api_key = os.getenv("DASHSCOPE_API_KEY")

audio_path = "resources/audio.mp3"

messages = [
    {
        "role": "user",
        "content": [
            {"audio": audio_path},
            {"text": "请识别这段音频的内容，转写成文字"},
        ],
    }
]

resp = dashscope.MultiModalConversation.call(
    model="qwen3.5-omni-plus",
    messages=messages,
    result_format="message",
)

if resp.status_code == HTTPStatus.OK:
    print(resp.output.choices[0].message.content)
else:
    raise RuntimeError(f"{resp.code}: {resp.message} request_id={resp.request_id}")


In [ ]:
print(f"mime_type: {mime_type}")

In [ ]:
# ============================================================
# 3. 流式调用
# ============================================================
print("识别结果:\n")
for chunk in model.stream([message]):
    print(chunk.content, end="", flush=True)

In [ ]:
# ============================================================
# 非流式版本：invoke() 一次性返回完整 AIMessage
# ============================================================
result = model.invoke([message])
print("识别结果:")
print(result.content)

In [ ]:
# 查看 AIMessage 的完整结构（含 token 用量等元数据）
from rich import print as rprint
rprint(result)

## 2.2.本地图片数据
有时候用户会上传图片数据，而不是图片的url地址。我们需要将图片数据转换成base64字符串，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。

```shell
uv add ipywidgets
```


然后，我们创建一个上传组件，用于模拟图片上传。


In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)

In [ ]:
print(uploader.value)

In [ ]:
# 读取图片，压缩并转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图 -> 字节
content_mv = uploaded_file["content"]
img_bytes = bytes(content_mv)

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
from langchain_core.messages import HumanMessage

# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image_url",
        "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"},
    },
    {"type": "text", "text": "给我讲讲图片"}
])

for chunk, metadata in agent.stream(
        {"messages": [multimodal_question]},
        stream_mode="messages"
):
    print(chunk.content, end="", flush=True)